In [ ]:
from pathlib import Path
import os, time, shutil, re

import matplotlib.pyplot as plt
import numpy as np
import scipy.io as sio
import xarray as xr
from importlib import reload

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "core").exists() and (REPO_ROOT.parent / "core").exists():
    os.chdir(REPO_ROOT.parent)
    REPO_ROOT = Path.cwd()

from core import config, data, infer, net, pipelines, simulation, train
from helpers import utils, utils_phantom

os.makedirs('figs', exist_ok=True)

## Read & organize the data

In [ ]:
reload(data); reload(pipelines); reload(utils); reload(utils_phantom); reload(net); reload(infer); reload(simulation)
reload(config); 

phantom_choice = '9T'
    
# Phantom 9.4T  # !!! See other choices in versions before 2026-Jan-17

config.Config.configure_LARG_phantom_preclinical_9T()
Larg_mM_GT, PH_GT = [50, 50, 50], [6, 5, 5.5]
signal = sio.loadmat('data/phantom9T_data.mat')['phantom'][..., None]
B1 = [0, 5, 5, 3, 3.75, 2.5, 1.75, 5.5, 6, 3.75, 5.75, 0.25, 3, 6, 4.5, 3.75, 3.5, 3.5, 0, 3.75, 6, 3.75, 4.75, 4.5, 4.25, 3.25, 5.25, 5.25, 0.25, 4.5, 5.25]
seq_df = data.read_sequence('larg', drop_first=True)
seq_df['B1_uT'] = B1[1:]
seq_df.reset_index(inplace=True, drop=True)
seq_df['dwRF_Hz'] = seq_df['ppm'] * (data.gamma/(2*np.pi) * data.data_config.B0_base) / 1e6
seq_df.loc[:, 'Tsat_ms'] = 3000
seq_df.loc[:, 'TR_ms'] = 4000

n_vials = len(Larg_mM_GT)   
plt.figure(figsize=(3, 3))
plt.imshow(signal[0, :, :, 0], cmap='gray', vmin=0, vmax=np.max(signal))
print(signal.shape)# seq_df_LARG

In [ ]:
centers, rois_nan, roi_mask_nans = utils_phantom.segment_vials_hough_seeded_watershed(signal[..., :1], visualize=True)
centers = [(y, x) for x, y in centers]    
#else:   # 2023-12-07 clinical - see versions before 2026-Jan-17

mask_nan = roi_mask_nans  


## Exploratory data analysis

### look at signals, validate measurement noise assumptions via GMM

In [ ]:
normed = (signal[:, :, :, 0]/signal[0, :, :, 0]*mask_nan).reshape((signal.shape[0], -1)).T

## EXP!
normed /= np.linalg.norm(normed, axis=1, keepdims=True) 

plt.figure(figsize=(20, 1))
for _signal in normed:
    plt.plot(
        np.arange(len(_signal)) + 0.01*np.random.randn(len(_signal)), 
        _signal, 
        'k-', linewidth=0.1, alpha=0.2, marker='.', markersize=3, markeredgecolor='r'
    )
ax1 = plt.gca()
ax1.set_xlabel("MRF cycle t", fontsize=10)
ax1.set_ylabel("S[t] (A.U.)", fontsize=10)

ax2 = ax1.twinx()
ax2.plot(B1, 'b', linewidth=1, label='B$_1$ ($\mu$T)')
ax2.set_ylabel('B1 (uT)', color='b')
ax2.tick_params(axis='y', colors='b')
ax2.spines['right'].set_color('b')

plt.xlim(3, 29)

In [ ]:
from sklearn.mixture import GaussianMixture

gmm = GaussianMixture(n_components=n_vials, random_state=0)
data4gmm = normed[np.isfinite(normed).all(axis=1)]
print(normed.shape, data4gmm.shape)
gmm.fit(data4gmm)

plt.figure(figsize=(18, 3))
for jj, mrf_it in enumerate([6, 7, 12]):
    plt.subplot(1, 3, jj+1)
    # Histogram
    counts, bins, _ = plt.hist(normed[:, mrf_it], bins=64, density=True, alpha=0.5, color='gray', label=f"S[{mrf_it}]/S[0] density")
    # Plot each Gaussian
    x = np.linspace(bins.min(), bins.max(), 256)
    for cluster_ind in range(n_vials):
        mu = gmm.means_[cluster_ind, mrf_it]
        sigma = np.sqrt(gmm.covariances_[cluster_ind, mrf_it, mrf_it])  # take variance from diagonal
        weight = gmm.weights_[cluster_ind]
        gaussian = weight * (1 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2)
        plt.plot(x, gaussian, label=f"$\mathcal{{N}}({mu:.3f}, {sigma:.4f})$") #f"$\mu$={mu:.2f}, $\sigma$={sigma:.3f}")
    plt.legend(loc='upper right', fontsize=10)
    plt.gca().get_yaxis().set_visible(False)
    gmm.means_.shape, gmm.weights_.shape, gmm.covariances_.shape

In [ ]:
# 7T
# 0.004/ 0.321, 0.0023/.255, 0.0018/0.228, 0.0012 / 0.166, 0.0015/0.198

```
The above Gaussian Mixture Modeling is validating assumptions - relatively gaussian noise with NRMSE of roughly 1.5-2%
```

In [ ]:
normed = normed[np.isfinite(normed).all(axis=1)]  # Remove rows with NaN values
normed.shape

In [ ]:
from scipy.stats import multivariate_normal

# Select two features (e.g., 6 and 7.  TODO try 12)

plt.figure(figsize=(20, 4))

for jj, (ix1, ix2) in enumerate([(6, 10), (6, 15), (10, 15)]):
    plt.subplot(1, 3, jj+1)

    # == DATA PLOT ==
    plt.scatter(normed[:, ix1], normed[:, ix2], s=3, color='b', label='Data', zorder=10 , alpha=0.5)

    # == DENSITY MODELING PLOT ==

    # Create a grid to evaluate the GMM
    x = np.linspace(normed[:, ix1].min(), normed[:, ix1].max(), 100)
    y = np.linspace(normed[:, ix2].min(), normed[:, ix2].max(), 100)
    X, Y = np.meshgrid(x, y)
    XX = np.column_stack([X.ravel(), Y.ravel()])

    # Compute the GMM marginal over the two variables
    Z = np.zeros(XX.shape[0])
    for k in range(gmm.n_components):
        mean = gmm.means_[k, [ix1, ix2]]
        cov = gmm.covariances_[k][np.ix_([ix1, ix2], [ix1, ix2])]
        weight = gmm.weights_[k]
        Z += weight * multivariate_normal(mean, cov).pdf(XX)
    Z = Z.reshape(X.shape)

    # Plot GMM contours
    plt.contour(X, Y, Z, levels=12, cmap='hot_r')# alpha=0.3)
    plt.xlabel(f'Scan {ix1}', fontsize=14)
    plt.ylabel(f'Scan {ix2}', fontsize=14)
    if jj == 0:
        plt.title('GMM 2D Marginal Contours over Data')
    plt.legend()



### Create data structure for fitting

In [ ]:
signal4fit = np.transpose(signal, (0,1,3,2))  # axial is 2nd  # [:, :, :, 0][:, :, None, :]
print(signal4fit.shape)

larg_ph_ds = data.SlicesFeed.from_args(
    shape=signal4fit.shape[1:], signal=signal4fit, 
    drop_first=False,  # the 9.4T is already dropped (30 scans)
    roi_mask_nans=mask_nan[:, None, :], seq_df=seq_df
    )

In [ ]:
plt.figure(figsize=(2,2))
plt.imshow(signal4fit[0,:,0,:], 'gray')
for idx, (y, x) in enumerate(centers[:6]):
    plt.text(x, y, str(idx), color='red', fontsize=18, fontweight='bold', ha='center', va='center')
print(centers)
print(Larg_mM_GT, PH_GT)

### Manual rough signal fitting using the known ground truth

In [ ]:
reload(utils); reload(pipelines)

H2O_mM = 110e3
Larg_spins = 3 
larg_ph_ds.fb_gt_T *= 0 
larg_ph_ds.kb_gt_T *= 0 
for roi, mM, PH in zip(rois_nan, Larg_mM_GT, PH_GT): # , k_GT):
    roi_tmp = np.copy(roi)
    roi_tmp[np.isnan(roi)] = 0 
    larg_ph_ds.fb_gt_T += roi_tmp[:,None,:] * mM * Larg_spins / H2O_mM
    # -- Heuristic for kb from pH    
    larg_ph_ds.kb_gt_T += roi_tmp[:,None,:] * (-30 + 450 * np.exp(1.1*(PH-5)))

sim_mode = 'expm_bmmat' if phantom_choice in ['7T', '9T'] else  'isar2_b'  # 'expm_bmmat' 'isar2_b'
tissue_params, nn_pred_signal_normed_np, nsignal_error = \
    pipelines.simple_infer_wrap(larg_ph_ds, nn_predictor=None, simulation_mode=sim_mode, visualize=True, sloi=0)

utils.draw_vial_sample_signals(larg_ph_ds, centers, nn_pred_signal_normed_np, sloi=0)

## Early stopping POC (w.o. neural net)


In [ ]:
import analyze_uncertainty as au

_x, _y = centers[1] 
_x -= 2 

renormalize = True 
# !! more "realistic" retrospective view where we do not know the true scaling (but giving ugly failure for low n with the phantom sequence lacking M0 "anchor")

loc_dict_res = 200
f_best_dotprod, k_best_dotprod, dict_signal, nrmse, df, dk = au.get_nrmse_grid(
    None, _x, _y, 0, 
    data_feed_mt=None, mt_tissue_param_est=tissue_params,
    data_feed_cest=larg_ph_ds, seq_name='larg', seq_df=seq_df,
    loc_dict_res=loc_dict_res
)       
n_meas = dict_signal.shape[0]
_signal = larg_ph_ds.measured_normed_T[:, _x, 0, _y][:,None,None]
_signal /= np.linalg.norm(_signal, axis=0) 
residuals = (dict_signal - _signal) 
# ! these are residuals of the "small" (norm=1) signal so their sum is nmse, not mse 
residuals_sq = residuals**2
cumsum_sqerr = np.cumsum(residuals_sq, axis=0) 
sigma_est = np.min(nrmse) / np.sqrt(n_meas)  # nrmse of measurement noise 
print(cumsum_sqerr.shape)

if renormalize:
    for jj in range(1, cumsum_sqerr.shape[0]+1):
        cumsum_sqerr[jj-1, :, :] = np.sum(( _signal[:jj]/np.linalg.norm(_signal[:jj]) - \
                                            dict_signal[:jj, :, :]/np.linalg.norm(dict_signal[:jj, :, :], axis=0) 
                                           )**2, axis=0)
print(cumsum_sqerr.shape)
cumul_log_likelihood = - cumsum_sqerr / (2 * sigma_est**2)
pdf = np.exp(cumul_log_likelihood)
pdf /= np.sum(pdf, axis=(1,2), keepdims=True)


In [ ]:
_x, _y

In [ ]:
from matplotlib.gridspec import GridSpec

def plot_single_voxel_shrinking(
    cumsum_sqerr, pdf,
    sample_voxel_CRs_indices = [4, 8, 12, 29] , _df=df, _dk=dk, n_meas=30, fig=None, fontsize=12
    ):

    fig = fig or plt.figure(figsize=(18, 4), constrained_layout=True)
    width_ratios = np.ones(len(sample_voxel_CRs_indices)+1)
    width_ratios[-2] = 0.5
    gs = GridSpec(1, len(sample_voxel_CRs_indices)+1, figure=fig, width_ratios=width_ratios)
    axes_top = np.array([fig.add_subplot(gs[0, i]) for i in range(len(sample_voxel_CRs_indices)+1)])
    
    f_prog_est = []
    k_prog_est = []
    CR_areas = []
    mM_CIs = []
    k_CIs = []
    for ii, jj in enumerate(sample_voxel_CRs_indices):
        ax = axes_top[ii] if ii < len(sample_voxel_CRs_indices)-1 else axes_top[-1]
        loc_nrmse = np.sqrt(n_meas*cumsum_sqerr[jj-1]/jj)
        min_nrmse = np.min(loc_nrmse)
        f_best = np.nanargmin(loc_nrmse) // loc_nrmse.shape[1]*_df*100
        k_best = np.nanargmin(loc_nrmse) %  loc_nrmse.shape[1]*_dk
        f_prog_est.append(f_best)
        k_prog_est.append(k_best)
        
        pdf_jj = pdf[jj-1].T
        pdf_jj_01 = pdf_jj/np.max(pdf[jj-1])
        imsh = ax.imshow(pdf_jj_01, origin='lower', aspect='auto', extent=[0, _df*100*loc_dict_res, 0, _dk*loc_dict_res], cmap='hot');     
        pdf01_th_arr = np.arange(0., 0.2, 0.004)
        _cdf = [np.sum(pdf_jj[pdf_jj_01 > th]) for th in pdf01_th_arr]  # np.arange(1.02, 1.25, 0.005):   # [1.04, 1.09, 1.2]:        
        pdf01_th = pdf01_th_arr[np.argmin(abs(np.array(_cdf) - 0.95))]
        cont = ax.contour(
                pdf_jj_01, 
                levels=[pdf01_th],
                colors=['magenta'],
                linewidths=[2], alpha=0.5, # 1 if jj in [20,21] else 0.5,
                extent=[0, _df*100*loc_dict_res, 0, _dk*loc_dict_res]
        )        
        ax.plot([f_best, f_best], [k_best, k_best],"kx", markersize=4)
        CR_area = np.sum(pdf_jj_01 > pdf01_th)  # in samples 
        CR_areas.append(CR_area)        
        #ax.set_xlim(0.06, 0.25)  !!
        ax.set_xlim(0.0, 0.25)
        ax.set_ylim(0, 1500)
        if ii>0:
            ax.set_yticklabels([])
        else:
            ax.set_yticklabels([f'{tick:.0f}' for tick in ax.get_yticks()], fontsize=fontsize+2)
        marg_f = np.sum(pdf_jj, axis=0)
        marg_f_cdf = np.cumsum(marg_f)
        marg_k = np.sum(pdf_jj, axis=1)
        marg_k_cdf = np.cumsum(marg_k)                
        f_975 = _df*100*np.searchsorted(marg_f_cdf, 97.5/100)
        f_025 = _df*100*np.searchsorted(marg_f_cdf, 2.5/100)                                     
        ax.plot((f_025, f_025), (0, _dk*loc_dict_res), 'w--', alpha=0.2)
        ax.plot((f_975, f_975), (0, _dk*loc_dict_res), 'w--', alpha=0.2)
        _GT_mM = 50 / H2O_mM * Larg_spins*100
        ax.plot((_GT_mM, _GT_mM), (0, _dk*loc_dict_res), 'y' ) #--', alpha=0.5, linewidth=2)    
        k_975 = _dk*np.searchsorted(marg_k_cdf, 97.5/100)
        k_025 = _dk*np.searchsorted(marg_k_cdf, 2.5/100)              
        ax.plot((0, _df*100*loc_dict_res), (k_025, k_025), 'w--', alpha=0.2)
        ax.plot((0, _df*100*loc_dict_res), (k_975, k_975), 'w--', alpha=0.2)            
        _mMticks = [0, 25, 50, 75, 100, 125]
        ax.set_xticks([_mMtick*Larg_spins/H2O_mM*100 for _mMtick in _mMticks])
        ax.set_xticklabels([f'{_mMtick}' for _mMtick in _mMticks], fontsize=fontsize+2)
        mM_best = (f_best/100) * H2O_mM / Larg_spins
        print('mM_best, k_best:', mM_best, k_best)
        mM_min, mM_max = (f_025/100) * H2O_mM / Larg_spins, (f_975/100) * H2O_mM / Larg_spins        
        mM_CI = mM_max - mM_min
        mM_CIs.append(mM_CI)
        k_CI = k_975 - k_025
        k_CIs.append(k_CI)                
        ax.text(0.05, 0.05, f"n={jj}", fontsize=fontsize+6, color='white', fontweight='bold', transform=ax.transAxes)
        ax.text(
            0.98, 0.98, # 0.14, 50, 
            f"[L-arg]={mM_min:.0f}-{mM_max:.0f} mM\nk={k_025:.0f}-{k_975:.0f} s$^{{-1}}$\nCR area={CR_area/4000*100:.2f} a.u.",
            color='white', transform=ax.transAxes, fontsize=fontsize+4, ha='right', va='top'
            ) 
        if ii == 0 :
            ax.set_ylabel('k$_{sw}$ (s$^{-1}$)', fontsize=fontsize+3)
        ax.set_xlabel('[L-arg] (mM)', fontsize=fontsize+2)
        axes_top[-2].axis('off')
        axes_top[-2].text(0.5, 0.5, '.  .  .', fontsize=40, fontweight='bold', ha='center', va='center', transform=axes_top[-2].transAxes)
        if ii==len(sample_voxel_CRs_indices)-1:
            cbar = plt.colorbar(imsh)
            cbar.set_label("Posterior probability\ndensity function (PDF)", fontsize=fontsize+2)
            cbar.ax.tick_params(labelsize=fontsize+2)
plot_single_voxel_shrinking(cumsum_sqerr, pdf, sample_voxel_CRs_indices = [3, 7, 10, 11, 30] )

In [ ]:
sig_norm2d = np.copy(larg_ph_ds.measured_normed_T[:, :, 0, :])
sig_norm2d[np.isnan(sig_norm2d)] = 0  # replace NaNs with 0s
sig_norm2d /= np.linalg.norm(sig_norm2d, axis=0)  # normalize to unit length
t_sig_norm2d = np.transpose(sig_norm2d, (1, 0, 2))
t_dict_signal = np.transpose(dict_signal, (1, 2, 0))

dict_signal.shape, sig_norm2d.shape
t_sig_norm2d[np.isnan(t_sig_norm2d)] = 0  # replace NaNs with 0s
dotprod = np.dot(t_dict_signal, t_sig_norm2d)  # dot product of the measured signal with the dictionary
dotprod.shape

# dotprod is shape (a, b, c, d)
argmax_flat = np.nanargmax(dotprod.reshape(-1, dotprod.shape[2], dotprod.shape[3]), axis=0)  # shape (c, d)
max_indices = np.array(np.unravel_index(argmax_flat, (dotprod.shape[0], dotprod.shape[1]))).transpose(1, 2, 0)  # shape (c, d, 2)
print(max_indices.shape)

_f_best = max_indices[:, :, 0] * df * 100 * larg_ph_ds.roi_mask_nans[:,0,:]
_k_best = max_indices[:, :, 1] * dk * larg_ph_ds.roi_mask_nans[:,0,:]
_mM = (_f_best / 100) * H2O_mM / Larg_spins 

_mM_final = np.copy(_mM)
_k_best_final = np.copy(_k_best)

vialmean_mM_final = np.zeros_like(_mM_final)
vialmean_k_best_final = np.zeros_like(_k_best_final)
k_GT_l =  [1067, 444, 607]  # CohenRapid2018 via QUESP
k_GT_map = np.zeros_like(_k_best_final)
for roi_nan, k_gt in zip(rois_nan, k_GT_l):
    roi_zero = np.copy(roi_nan)
    roi_zero[np.isnan(roi_zero)] = 0  # replace NaNs with 0s
    vialmean_mM_final += np.nanmean(roi_nan * _mM_final) * roi_zero
    vialmean_k_best_final += np.nanmean(roi_nan * _k_best_final) * roi_zero
    k_GT_map += k_gt * roi_zero

In [ ]:
from matplotlib.gridspec import GridSpec

def plot_progressive_maps(
        t_sig_norm2d, t_dict_signal, do_vialwise_mape=True, do_GT_mape = True, mM_GT=50, # !! caution hardcoded
        MAE_nmeas_list = [4, 8, 12, 29], _df=df, _dk=dk, fig=None, fontsize=12, xy=None
    ):   
    ''' Note: in phantom we don't use any T1,T2,B0,B1 maps so low-dimensional dictionary is fine (same for any voxel)
    '''
    fig = fig or plt.figure(figsize=(18, 8), constrained_layout=True)     
    width_ratios = np.ones(len(MAE_nmeas_list)+1)
    width_ratios[-2] = 0.5
    gs = GridSpec(2, len(MAE_nmeas_list)+1, figure=fig, width_ratios=width_ratios)
    axes = np.array([[fig.add_subplot(gs[row, i]) for i in range(len(MAE_nmeas_list)+1)] for row in range(2)])
    
    t0 = time.time()
    for ii, nn in enumerate(MAE_nmeas_list): 
        tmp = ii if ii < len(MAE_nmeas_list)-1 else len(MAE_nmeas_list)        
        _ax1, _ax2 = axes[:, int(tmp)] 
        
        _dict_nn = np.copy(t_dict_signal[..., :nn])        
        _sig_nn = np.copy(t_sig_norm2d[..., :nn, :])
        if renormalize:
            _dict_nn /= np.linalg.norm(_dict_nn, axis=-1, keepdims=True)
            _sig_nn /= np.linalg.norm(_sig_nn, axis=-2, keepdims=True)                                 
            dotprod = np.dot(_dict_nn, _sig_nn)  # dot product of the measured signal with the dictionary        
            argmax_flat = np.nanargmax(dotprod.reshape(-1, dotprod.shape[2], dotprod.shape[3]), axis=0)  # shape (c, d) # ..last axis of a and second last of b
        else:        
            dotprod = np.dot(_dict_nn, _sig_nn)  # dot product of the measured signal with the dictionary        
            diffnorm = np.linalg.norm(_dict_nn, axis=-1)[..., None, None]**2 + np.linalg.norm(_sig_nn, axis=-2)[None, None,...]**2 - 2*dotprod        
            argmax_flat = np.nanargmin(diffnorm.reshape(-1, dotprod.shape[2], dotprod.shape[3]), axis=0)  # shape (c, d)                
        max_indices = np.array(np.unravel_index(argmax_flat, (dotprod.shape[0], dotprod.shape[1]))).transpose(1, 2, 0)  # shape (c, d, 2)
        
        _f_best = max_indices[:, :, 0] * _df * 100 * larg_ph_ds.roi_mask_nans[:,0,:]
        _mM = (_f_best / 100) * H2O_mM / Larg_spins  # convert to mM    
        _k_best = max_indices[:, :, 1] * _dk  * larg_ph_ds.roi_mask_nans[:,0,:]    
        print('best mM, k:', _mM[_x, _y], _k_best[_x, _y])     
        # ! MAPE nuance:  ideally vs GT, here taking the final mM and k_best either voxelwise or vialwise        
        mM_MAPE = 100 * np.nanmean(np.abs(_mM - _mM_final) / (1e-6 + _mM_final))
        k_MAPE = 100 * np.nanmean(np.abs(_k_best - _k_best_final) / (1e-6 + _k_best_final))
        _ax1.text(x=0.02, y=.96, transform=_ax1.transAxes, s=f'MAPE={mM_MAPE:.1f}%', fontsize=fontsize+2, va='top')
        _ax2.text(x=0.02, y=.96, transform=_ax2.transAxes, s=f'MAPE={k_MAPE:.1f}%', fontsize=fontsize+2, va='top')
        
        if do_vialwise_mape:
            mM_MAPE = 100 * np.nanmean(np.abs(_mM - vialmean_mM_final) / (1e-6 + vialmean_mM_final))            
            k_MAPE = 100 * np.nanmean(np.abs(_k_best - vialmean_k_best_final) / (1e-6 + vialmean_k_best_final))            
            _ax1.text(x=0.02, y=.88, transform=_ax1.transAxes, s=f'MAPE$_1$={mM_MAPE:.1f}%', fontsize=fontsize+2, va='top')
            _ax2.text(x=0.02, y=.88, transform=_ax2.transAxes, s=f'MAPE$_1$={k_MAPE:.1f}%', fontsize=fontsize+2, va='top')
        if do_GT_mape:
            mM_MAPE = 100 * np.nanmean(np.abs(_mM - mM_GT) / (1e-6 + mM_GT))
            _ax1.text(x=0.02, y=.8, transform=_ax1.transAxes, s=f'MAPE$_{{GT}}$={mM_MAPE:.1f}%', fontsize=fontsize+2, va='top')
            k_MAPE = 100 * np.nanmean(np.abs(_k_best - k_GT_map) / (1e-6 + k_GT_map))
            _ax2.text(x=0.02, y=.8, transform=_ax2.transAxes, s=f'MAPE$_{{GT}}$={k_MAPE:.1f}%', fontsize=fontsize+2, va='top')            
        
        _data = _mM
        print(_mM.shape)
        xmin, xmax, ymin, ymax = 5, 53, 7, 53  # crop for better visualization   
        _img = _ax1.imshow(_data[xmin: xmax, ymin: ymax], vmin=0, vmax=100, origin='lower') 
        # y-axis inverted to have origin at bottom left but it still reflects first axis of array..
        
        cbar = utils.cbarhist(_img, _data, _ax1, bins=32) # , bins=np.arange(0.1, 0.2, _df*100)); 
        if nn==MAE_nmeas_list[-1]:  # work around a bug on second row by aligning both
            cbar.set_label('[L-arg] (mM)', fontsize=fontsize+2)
            cbar.ax.tick_params(labelsize=fontsize+2)
        else:
            cbar.set_ticks([]) 

        if xy is not None:
            _ax1.plot([xy[1]-ymin], [xy[0]-xmin], marker='s', markerfacecolor='none', markeredgecolor='violet', markersize=7, markeredgewidth=3) #, linewidths=2)        
        
        _ax1.set_xticks([]);  _ax1.set_yticks([])
        _ax2.set_xticks([]); _ax2.set_yticks([])
            
        _data = (_k_best * larg_ph_ds.roi_mask_nans[:,0,:])
        _img = _ax2.imshow(_data[xmin: xmax, ymin: ymax], vmin=0, vmax=1200, cmap='magma', origin='lower')         
        print(_data[xmin: xmax, ymin: ymax].shape)
        
        cbar = utils.cbarhist(_img, _data, _ax2, bins=32) #, bins=np.arange(vmin=0, vmax=1000, _dk)); 
        if nn==MAE_nmeas_list[-1]:  
            cbar.set_label('k$_{sw}$ (s$^{-1})$', fontsize=fontsize+3)
            cbar.ax.tick_params(labelsize=fontsize+2)
        else:
            cbar.set_ticks([]) 
        _ax2.text(x=0.01, y=-0.01, va='top', ha='left', transform=_ax2.transAxes, s=f'n={nn}', fontweight='bold', fontsize=fontsize+5)
        axes[0, -2].axis('off'); axes[1, -2].axis('off')
        axes[0, -2].text(0.5, 0.5, '.  .  .', fontsize=40, fontweight='bold', ha='center', va='center', transform=axes[0, -2].transAxes)
                
        

In [ ]:
def slap_label_on_container(container, label_text, loc=(-0.01, 1.04), color='black'):    
    ghost_ax = container.add_axes([0, 0, 1, 1], frameon=False)        
    ghost_ax.set_xticks([]); ghost_ax.set_yticks([])        
    ghost_ax.text(
        loc[0], loc[1], label_text, transform=ghost_ax.transAxes, 
        fontsize=18, fontweight='bold', va='top', ha='left', 
        color=color
    )  

# FINAL FIGURE (Fig. 10)

In [ ]:
fig = plt.figure(figsize=(18, 13), constrained_layout=True)
subfigs = fig.subfigures(3, 1, height_ratios=[1.2, 0.6, 0.5], hspace=0.08)

ind_list = [3, 7, 10, 11, 30]
plot_progressive_maps(
    t_sig_norm2d, t_dict_signal, xy=(_x, _y),
    fig=subfigs[0], MAE_nmeas_list = ind_list
)
plot_single_voxel_shrinking(cumsum_sqerr, pdf, fig=subfigs[1], fontsize=13, sample_voxel_CRs_indices=ind_list)

axbot = subfigs[2].add_subplot(1,1,1)
l4, = axbot.plot(range(1, 31), B1[1:], 'bX-', markersize=10, linewidth=1, label='B$_1$ ($\mu$T)')
axbot.set_ylabel('B1 (uT)', color='b', fontsize=17) #, font='bold')
axbot.tick_params(axis='y', colors='b')
axbot.spines['left'].set_color('b')
axbot.spines['left'].set_linewidth(4)
axbot.set_xlabel('Number of scans acquired (n)', fontsize=18)
axbot.set_xticklabels(axbot.get_xticklabels(), fontsize=18, fontweight='bold')
axbot.set_yticklabels(axbot.get_yticklabels(), fontsize=18, fontweight='bold')
axbot_twin = axbot.twinx()
l1, = axbot_twin.plot(range(1, 31), gmm.means_[0]/np.max(gmm.means_[0]), 'kd:', markerfacecolor='peachpuff', markersize=7, markeredgewidth=1, linewidth=1, label='Vial 1 (mean)')
l2, = axbot_twin.plot(range(1, 31), gmm.means_[1]/np.max(gmm.means_[1]), 'ks:', markerfacecolor='pink', markersize=7, markeredgewidth=1, linewidth=1, label='Vial 2 (mean)')
l3, = axbot_twin.plot(range(1, 31), gmm.means_[2]/np.max(gmm.means_[2]), 'ko:', markerfacecolor='paleturquoise', markersize=7, markeredgewidth=1, linewidth=1, label='Vial 3 (mean)')
axbot_twin.set_yticklabels(axbot_twin.get_yticklabels(), fontsize=18)
axbot_twin.set_ylabel('Normed signal\n S[t]/max(S)    ', color='k', fontsize=16)
plt.legend(handles=[l1, l2, l3, l4], loc='center right', fontsize=15, framealpha=0.85, facecolor='lightgray')

slap_label_on_container(subfigs[0], '(a)', loc=(-0.01, 1.03))
slap_label_on_container(subfigs[1], '(b)', loc=(-0.01, 1.02))
slap_label_on_container(subfigs[2], '(c)', loc=(-0.01, 1.02))

plt.savefig('figs/phantom_seq_LOWRES.png', bbox_inches='tight', dpi=100, format='png')
plt.savefig('figs/phantom_seq.png', bbox_inches='tight', dpi=300, format='png')
plt.savefig('figs/phantom_seq.svg', bbox_inches='tight', dpi=300, format='svg')

## Run PdVAE fitting



In [ ]:
larg_ph_ds.shape

In [ ]:
model_state, hist, net_kwargs = train.train(larg_ph_ds, lr=0.01, steps=2000, simulation_mode=sim_mode, patience=1000)

plt.figure(figsize=(10, 2))
plt.semilogy(hist['est_loss_history'], label='loss')
plt.ylim(1, 30)
plt.grid(True, which='both', linestyle='--', linewidth=0.5)
plt.minorticks_on()

!! Multiple runs (inc for 3T/7T phantoms) shown in notebooks before 2026-Jan-18

In [ ]:
reload(utils); reload(pipelines)

predictor = net.state2predictor(model_state)
larg_ph_ds.ds = 1
tissue_params, nn_pred_signal_normed_np, nsignal_error = \
    pipelines.simple_infer_wrap(larg_ph_ds, nn_predictor=predictor, visualize=True, sloi=0)
    
plt.figure(figsize=(10, 3))
_ax = plt.subplot(121)
_data = H2O_mM/Larg_spins * tissue_params['fb_T'][:, 0, :]
_img = plt.imshow(_data, vmin=0, vmax=120); 
cbar = utils.cbarhist(_img, _data, _ax)
cbar.set_label('[L-arg] (mM)')
_ax = plt.subplot(122)
_data = tissue_params['kb_T'][:, 0, :]
_img = plt.imshow(_data, vmin=0, vmax=1500);  
cbar = utils.cbarhist(_img, _data, _ax)
cbar.set_label('k$_s$ (s$^{-1}$)')

kvial = [np.nanmean(tissue_params['kb_T'][:, 0, :]*roi) for roi in rois_nan]
plt.figure(figsize=(6, 2))
plt.plot(PH_GT, kvial, 'd', label='k$_s$ from vials', markerfacecolor='none', markeredgecolor='k', markersize=8)
for PH, roi in zip(PH_GT, rois_nan):
    kvals = (tissue_params['kb_T'][:, 0, :]*roi).flatten()
    fvals = (tissue_params['fb_T'][:, 0, :]*roi).flatten()
    print(f'PH={PH}: f_s mean={100*np.nanmean(fvals):.3f}%, k_s mean={np.nanmean(kvals):.2f} s^-1')
    kvals = kvals[np.isfinite(kvals)]
    plt.plot(PH + 1e-4*np.random.randn(len(kvals)), kvals)
phgrid = np.arange(3.7, 6.3, 0.1)
plt.plot(phgrid, 80 + 300 * np.exp(1.1*(phgrid-5)), '--', alpha=0.5, linewidth=3,)
plt.xlabel('PH'); plt.ylabel('k$_s$ (s$^{-1}$)')

In [ ]:
import analyze_uncertainty as au;  reload(au)

f_est, k_est, cov_nnpred_scaled, f_sigma, k_sigma, height, width, angle = \
    au.extract_nn_estimate_of_posterior(tissue_params, larg_ph_ds.shape, is_cest=True)

In [ ]:
100*np.array(Larg_mM_GT) * Larg_spins / H2O_mM, PH_GT, Larg_mM_GT

In [ ]:
points = [cc for cc in centers]
points += [(x+2, y+0  ) for (x,y) in centers]
points += [(x-2, y+0) for (x,y) in centers]
points += [(x, y+2) for (x,y) in centers]
points += [(x, y-2) for (x,y) in centers]
labels = [0,1,2,3,4,5]*5 

all_points = []
all_labels = []
for cind, cc in enumerate(centers):
    for ii in range(-4, 4):
        for jj in range(-4, 4):
            if not np.isnan(roi_mask_nans[cc[0]+ii, cc[1]+jj]):
                all_points += [(cc[0]+ii, cc[1]+jj)]
                all_labels += [cind]
            else:
                pass

z = 5  if phantom_choice == '3T' else 0 
max_k = 800 if phantom_choice == '3T' else 1500
flims = [0, 0.3] if phantom_choice == '3T' else [0.1, 0.2]
klims = [0, max_k] if phantom_choice in ['3T', '7T'] else [200, 1200]  # else 1500# [300, 700]  #[0, max_k]

reload(au)
au.demonstrate_ellipses(
    all_points[::5], f_est[:,z,:], k_est[:,z,:], 
    width[:,z,:], height[:,z,:], angle[:,z,:],
    all_labels[::5], is_mt=False, xlim=flims, ylim=klims,
    figsize=(15,5)
)

for c, x in zip('cmykrgb', 100*np.array(Larg_mM_GT) * Larg_spins / H2O_mM): 
    rr = np.random.random()*0.005
    plt.plot([x+rr, x+rr], [0, max_k], c+'-', alpha=0.3)

In [ ]:
reload(au)

def plot_sample_posteriors(points, data_feed_cest=None, seq_name='mt', seq_df=None, _z=0, xlim=None, ylim=None, packed=False):
    if packed:
        plt.figure(figsize=(16, 16))
    for jj, (_x, _y) in enumerate(points):  #[::3] [[8, 24], [13, 24], [8, 9], [13, 9]]):    
        f_best_dotprod, k_best_dotprod, dict_signal, nrmse, _df, _dk = au.get_nrmse_grid(
            None, _x, _y, _z, 
            data_feed_mt=None, mt_tissue_param_est=tissue_params,
            data_feed_cest=data_feed_cest, seq_name=seq_name, seq_df=seq_df,
            loc_dict_res=200
        )    
        maha1, maha2, posterior_cov, CR_area, CIk_x_CIf, _ = au.viz_posteriors(
            f_est[_x, _z, _y], k_est[_x, _z, _y], cov_nnpred_scaled[_x, _z, _y], 
            width[_x, _z, _y], height[_x, _z, _y], angle[_x, _z, _y],
            f_best_dotprod, k_best_dotprod, nrmse, _df, _dk,        
            is_cest=True, 
            fontsize=14, figsize=[7, 5], # figsize=(6, 4),  # good w.o. marginals
            do_marginals=True, show_text=True, show_NN=True, do_legend=False,            
            ax=plt.gca() if packed else None, 
            do_cbar=not packed,
            xlim=xlim, ylim=ylim, loc_dict_res=200
            )    
        
        print(f"maha1={maha1:.1f}, maha2={maha2:.1f}")
        plt.tight_layout()
        print("min(nrmse)", np.min(nrmse))
    
    print(f'Done {jj}')
    return dict_signal, nrmse, _df, _dk

# Returning data for last tested voxel for further drill-down
dict_signal, nrmse, _df, _dk = \
        plot_sample_posteriors(points, larg_ph_ds, seq_name='larg', seq_df=seq_df, xlim=[0.0, 0.25], ylim=[0, max_k])#, packed=True)

# Stats for table


In [ ]:
larg_ph_ds.shape

In [ ]:
np.nanmean(tissue_params['kc_T'])

In [ ]:
reload(au); reload(au.metrics); reload(au.viz)
au.au_config.nrmse_loc_dict_res = 200

def meshgrid_xy_pairs(shape):
    x, y = np.meshgrid(np.arange(shape[0]), np.arange(shape[1]), indexing='ij')
    xy_pairs = np.stack([x, y], axis=-1).reshape(-1, 2)
    return xy_pairs.tolist()

f_est, k_est, cov_nnpred_scaled, f_sigma, k_sigma, height, width, angle = \
    au.extract_nn_estimate_of_posterior(tissue_params, larg_ph_ds.shape, is_cest=True)
    
xy_points = meshgrid_xy_pairs(larg_ph_ds.shape[0::2])

do_all_points = True  # set false to reduce to <1min run
if not do_all_points:
    xy_points = xy_points[::10]
    
mahas1, mahas2, CR_areas, nnCR_areas, good_xy_points, \
    modes_of_exact_posterior, means_of_exact_posterior, covs_of_exact_posterior, timings, \
    exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs, extra_metrics = \
    au.compare_posteriors(
        xy_points, data_feed_mt=larg_ph_ds, mt_tissue_params_est=tissue_params, is_cest=True,
        data_feed_cest=larg_ph_ds, cest_tissue_params_est=tissue_params, seq_df=seq_df, seq_name='larg',
        voxels2sample=1000, visualize=False
    )   


In [ ]:
len(mahas1)

In [ ]:
results_folder = './figs/phantom'
os.makedirs(results_folder, exist_ok=True)

os.makedirs(results_folder+'/intermediates', exist_ok=True)

# 0, 1, 2
vial_labels = [[ii for ii,rr in enumerate(rois_nan) if not np.isnan(rr[x,y])][0] for [x,y] in good_xy_points]

np.savez_compressed(
    results_folder+'/intermediates/UQ_stats_allpoints.npz',    
    mahas1=mahas1, mahas2=mahas2,
    CR_areas=CR_areas,  nnCR_areas=nnCR_areas,
    good_xy_points=good_xy_points, timings=timings,
    f_est=f_est, k_est=k_est, cov_nnpred_scaled=cov_nnpred_scaled,
    covs_of_exact_posterior=covs_of_exact_posterior, modes_of_exact_posterior=modes_of_exact_posterior, means_of_exact_posterior=means_of_exact_posterior,    
    exact_posterior_min_nrmse_percs=exact_posterior_min_nrmse_percs, nn_posterior_min_nrmse_percs=nn_posterior_min_nrmse_percs,
    vial_labels=vial_labels,
    **extra_metrics
)

np.savez(
    f'{results_folder}/intermediates/amide_tissue_params_est.npz',    
    nn_pred_signal_normed_np=nn_pred_signal_normed_np,    
    **tissue_params
)    

In [ ]:
# 0, 1, 2
vial_labels = [[ii for ii,rr in enumerate(rois_nan) if not np.isnan(rr[x,y])][0] for [x,y] in good_xy_points]

for label, [x,y] in zip(vial_labels, good_xy_points):
    print(label, [rr[x,y] for rr in rois_nan])

In [ ]:
plt.hist(mahas1, bins=50, range=(0, 6), histtype='step')
plt.hist(mahas2, bins=50, range=(0, 6, ), histtype='step')

### Removed content 

See versions before 19-20.01.2026 for lots of alternative analyses and visualizations
